# Flatmate RL Synthetic SFT Training And Evaluation

This notebook generates heuristic SFT data, trains a local causal LM, evaluates both the base model and the SFT model with environment rollouts, and plots task-wise comparisons.

Default split:
- Train: 100 seeded episodes per task id
- Test: 20 held-out seeded episodes per task id

Install optional ML dependencies before running training/evaluation cells if they are not already available:

```bash
pip install -e .[dev]
pip install torch transformers datasets accelerate pandas matplotlib
```

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd() / "flatmate_rl", Path("/content/flatmate_rl")]
    candidates.extend(Path.cwd().parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "server").exists():
            return candidate.resolve()
    raise RuntimeError("Could not find the flatmate_rl repo. In Colab, run: !git clone https://github.com/kushal0412/flatmate_rl.git")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sft_synthetic import generate_dataset, evaluate_policy, load_hf_policy
from server.scenarios import SCENARIOS

BASE_MODEL = os.getenv("FLATMATE_BASE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
DATA_DIR = Path("data/sft_notebook")
OUTPUT_DIR = Path("outputs/flatmate-sft-notebook")
RESULTS_DIR = OUTPUT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SCENARIO_IDS = sorted(SCENARIOS.keys())
TRAIN_SEEDS = list(range(0, 100))
TEST_SEEDS = list(range(1000, 1020))
MAX_ENV_STEPS = 28
STRICT_EVAL = True

print(f"Base model: {BASE_MODEL}")
print(f"Tasks: {len(SCENARIO_IDS)}")
print(f"Train episodes per task: {len(TRAIN_SEEDS)}")
print(f"Test episodes per task: {len(TEST_SEEDS)}")

## 1. Generate Synthetic SFT Data

The split is scenario-balanced: the same 100 training seeds and 20 test seeds are rolled out for every task id.

In [ ]:
manifest = generate_dataset(
    output_dir=DATA_DIR,
    scenario_ids=SCENARIO_IDS,
    train_seeds=TRAIN_SEEDS,
    eval_seeds=TEST_SEEDS,
    max_steps=MAX_ENV_STEPS,
    strict_eval=STRICT_EVAL,
)
print(json.dumps(manifest, indent=2, sort_keys=True))

In [ ]:
import pandas as pd

def read_jsonl(path: Path) -> pd.DataFrame:
    return pd.read_json(path, lines=True)

train_rows = read_jsonl(DATA_DIR / "train.jsonl")
test_rows = read_jsonl(DATA_DIR / "eval.jsonl")

split_counts = pd.concat(
    [
        train_rows.groupby("scenario_id").size().rename("train_examples"),
        test_rows.groupby("scenario_id").size().rename("test_examples"),
    ],
    axis=1,
).fillna(0).astype(int)
split_counts

## 2. Train The SFT Model

This cell uses the same trainer utilities as `train_sft.py`. It masks the prompt and trains only on the assistant JSON action.

In [ ]:
from train_sft import (
    build_collator,
    build_tokenizer_mapper,
    load_sft_dataset,
    require_training_deps,
    training_arguments,
)

# Adjust these for your hardware.
MAX_LENGTH = 4096
EPOCHS = 2.0
BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-5
BF16 = False
FP16 = False

torch, Dataset, DatasetDict, AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments = require_training_deps()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if BF16 else torch.float16 if FP16 else None,
)
model.config.pad_token_id = tokenizer.pad_token_id

dataset = load_sft_dataset(Dataset, DatasetDict, DATA_DIR / "train.jsonl", DATA_DIR / "eval.jsonl")
tokenized = dataset.map(
    build_tokenizer_mapper(tokenizer, MAX_LENGTH),
    remove_columns=dataset["train"].column_names,
)

class Args:
    output_dir = OUTPUT_DIR
    epochs = EPOCHS
    batch_size = BATCH_SIZE
    eval_batch_size = EVAL_BATCH_SIZE
    gradient_accumulation_steps = GRAD_ACCUM
    learning_rate = LEARNING_RATE
    logging_steps = 10
    eval_steps = 50
    save_steps = 100
    save_total_limit = 2
    warmup_ratio = 0.03
    weight_decay = 0.0
    bf16 = BF16
    fp16 = FP16

trainer = Trainer(
    model=model,
    args=training_arguments(TrainingArguments, Args),
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["eval"],
    data_collator=build_collator(tokenizer),
)

trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Saved SFT model to {OUTPUT_DIR}")

# Free memory before loading checkpoints for rollout evaluation.
import gc
del trainer, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 3. Evaluate Base Model And SFT Model

Both evaluations use real environment rollouts on the 20 held-out seeds per task. The model must emit valid JSON actions at every step.

In [ ]:
def evaluate_hf_checkpoint(label: str, model_path: str) -> pd.DataFrame:
    action_fn = load_hf_policy(model_path=model_path, max_new_tokens=256, temperature=0.0)
    result = evaluate_policy(
        policy_name=label,
        scenario_ids=SCENARIO_IDS,
        seeds=TEST_SEEDS,
        max_steps=MAX_ENV_STEPS,
        action_fn=action_fn,
        strict_eval=STRICT_EVAL,
    )
    with (RESULTS_DIR / f"{label.lower().replace(' ', '_')}.json").open("w", encoding="utf-8") as handle:
        json.dump(result, handle, indent=2, sort_keys=True)
    action_fn = None
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass
    df = pd.DataFrame(result["episodes"])
    df["policy"] = label
    return df

base_eval = evaluate_hf_checkpoint("Base model", BASE_MODEL)
sft_eval = evaluate_hf_checkpoint("SFT model", str(OUTPUT_DIR))
episode_results = pd.concat([base_eval, sft_eval], ignore_index=True)
episode_results.to_csv(RESULTS_DIR / "episode_results.csv", index=False)
episode_results.head()

## 4. Task-Wise Metrics And Performance Difference

Metrics:
- `success_rate`: fraction of episodes completed without violations
- `avg_score`: mean environment total reward
- `avg_steps_to_completion`: mean number of environment steps used

In [ ]:
summary = (
    episode_results.groupby(["policy", "scenario_id"])
    .agg(
        success_rate=("success", "mean"),
        avg_score=("total_reward", "mean"),
        avg_steps_to_completion=("steps", "mean"),
        parse_errors=("parse_errors", "sum"),
        episodes=("seed", "count"),
    )
    .reset_index()
)
summary.to_csv(RESULTS_DIR / "task_summary.csv", index=False)
summary

In [ ]:
base_summary = summary[summary["policy"] == "Base model"].set_index("scenario_id")
sft_summary = summary[summary["policy"] == "SFT model"].set_index("scenario_id")

diff = pd.DataFrame(index=SCENARIO_IDS)
diff["success_rate_delta"] = sft_summary["success_rate"] - base_summary["success_rate"]
diff["avg_score_delta"] = sft_summary["avg_score"] - base_summary["avg_score"]
diff["avg_steps_delta"] = sft_summary["avg_steps_to_completion"] - base_summary["avg_steps_to_completion"]
diff = diff.reset_index(names="scenario_id")
diff.to_csv(RESULTS_DIR / "sft_minus_base.csv", index=False)
diff

## 5. Plot Base Versus SFT Comparisons

In [ ]:
import matplotlib.pyplot as plt

metrics = [
    ("success_rate", "Success Rate", "higher is better"),
    ("avg_score", "Average Score", "higher is better"),
    ("avg_steps_to_completion", "Average Steps To Completion", "lower is better"),
]

fig, axes = plt.subplots(len(metrics), 1, figsize=(16, 18), sharex=True)
for ax, (metric, title, note) in zip(axes, metrics):
    pivot = summary.pivot(index="scenario_id", columns="policy", values=metric).loc[SCENARIO_IDS]
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(f"{title} ({note})")
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="best")

plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plot_path = RESULTS_DIR / "base_vs_sft_task_metrics.png"
plt.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved plot to {plot_path}")

In [ ]:
overall = (
    episode_results.groupby("policy")
    .agg(
        success_rate=("success", "mean"),
        avg_score=("total_reward", "mean"),
        avg_steps_to_completion=("steps", "mean"),
        parse_errors=("parse_errors", "sum"),
        episodes=("seed", "count"),
    )
    .reset_index()
)
overall.to_csv(RESULTS_DIR / "overall_summary.csv", index=False)
overall